##### Copyright 2026 Google LLC.

In [21]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini API: Python으로 함수 호출하기

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Function_calling.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

 함수 호출(Function calling)을 사용하면 개발자가 코드에서 함수 설명을 생성한 다음 해당 설명을 요청의 언어 모델에 전달할 수 있습니다. 모델의 응답에는 설명과 일치하는 함수 이름과 호출할 인수가 포함됩니다. 함수 호출을 통해 생성형 AI 애플리케이션에서 함수를 도구로 사용할 수 있으며, 단일 요청에서 하나 이상의 함수를 정의할 수 있습니다.

이 노트북은 시작하는 데 도움이 되는 코드 예제를 제공합니다. 문서의 [빠른 시작](https://ai.google.dev/gemini-api/docs/function-calling#python)도 함수 호출을 이해하기 위한 좋은 출발점입니다.

## 설정

### 의존성 설치

In [22]:
%pip install -qU 'google-genai>=1.0.0'

### API 키 설정

다음 셀을 실행하려면 API 키를 `GOOGLE_API_KEY`라는 이름의 Colab 시크릿에 저장해야 합니다. API 키가 없거나 Colab 시크릿 생성 방법이 궁금하다면 [인증 ![image](https://storage.googleapis.com/generativeai-downloads/images/colab_icon16.png)](../quickstarts/Authentication.ipynb) 빠른 시작을 참고하세요.

In [23]:
from google import genai
from google.colab import userdata

GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
client = genai.Client(api_key=GOOGLE_API_KEY)

### 모델 선택

함수 호출은 모든 Gemini 모델에서 작동합니다.

In [24]:
MODEL_ID = "gemini-3-flash-preview" # @param ["gemini-2.5-flash-lite", "gemini-2.5-flash", "gemini-2.5-pro", "gemini-2.5-flash-preview", "gemini-3.1-flash-lite-preview", "gemini-3.1-pro-preview"] {"allow-input":true, isTemplate: true}

## 함수를 도구로 설정하기

함수 호출을 사용하려면 [`GenerativeModel`](https://ai.google.dev/api/python/google/generativeai/GenerativeModel)을 만들 때 `tools` 파라미터에 함수 목록을 전달하세요. 모델은 함수 이름, docstring, 파라미터, 파라미터 타입 어노테이션을 사용하여 프롬프트에 가장 적합한 함수가 필요한지 결정합니다.

> 중요: SDK는 함수 파라미터 타입 어노테이션을 API가 이해하는 형식(`genai.types.FunctionDeclaration`)으로 변환합니다. API는 제한된 파라미터 유형만 지원하며, Python SDK의 자동 변환은 그 중 일부만 지원합니다: `AllowedTypes = int | float | bool | str | list['AllowedTypes'] | dict`


**예제: 조명 시스템 함수**

다음은 가상의 조명 시스템을 제어하는 3개의 함수입니다. docstring과 타입 힌트에 주목하세요.

In [25]:
def enable_lights():
    """Turn on the lighting system."""
    print("LIGHTBOT: Lights enabled.")


def set_light_color(rgb_hex: str):
    """Set the light color. Lights must be enabled for this to work."""
    print(f"LIGHTBOT: Lights set to {rgb_hex}.")

def stop_lights():
    """Stop flashing lights."""
    print("LIGHTBOT: Lights turned off.")

light_controls = [enable_lights, set_light_color, stop_lights]
instruction = """
  You are a helpful lighting system bot. You can turn
  lights on and off, and you can set the color. Do not perform any
  other tasks.
"""

## 채팅을 통한 기본 함수 호출

함수 호출은 다중 턴 대화에 자연스럽게 맞습니다. Python SDK의 `ChatSession(client.chats.create(...))`은 대화 기록을 자동으로 처리하므로 이에 이상적입니다.

또한 `ChatSession`은 `automatic_function_calling` 기능(기본적으로 활성화)을 통해 함수 호출 실행을 단순화합니다. 이에 대해서는 나중에 더 자세히 살펴보겠습니다. 우선 모델이 함수를 호출하기로 결정하는 기본 상호작용을 살펴보겠습니다.

In [26]:
chat = client.chats.create(
    model=MODEL_ID,
    config={
        "tools": light_controls,
        "system_instruction": instruction,
        # automatic_function_calling defaults to enabled
    }
)

response = chat.send_message("It's awful dark in here...")

print(response.text)

LIGHTBOT: Lights enabled.
No problem, I can turn the lights on for you.



## 함수 호출 및 실행 기록 검토

백그라운드에서 발생한 일을 이해하기 위해 채팅 기록을 검토할 수 있습니다.

`Chat.history` 속성은 사용자와 Gemini 모델 간의 대화를 시간순으로 기록합니다. `Chat.get_history()`를 사용하여 기록을 가져올 수 있습니다. 대화의 각 턴은 다음 정보를 포함하는 `genai.types.Content` 객체로 표현됩니다:

**역할(Role)**: 콘텐츠가 "user"에서 비롯되었는지 "model"에서 비롯되었는지 식별합니다.

**파트(Parts)**: 메시지의 개별 구성 요소를 나타내는 `genai.types.Part` 객체 목록. 텍스트 전용 모델에서 이 파트는 다음과 같습니다:

* **텍스트(Text)**: 일반 텍스트 메시지.
* **함수 호출(Function Call, genai.types.FunctionCall)**: 특정 함수를 제공된 인수로 실행하도록 모델이 요청.
* **함수 응답(Function Response, genai.types.FunctionResponse)**: 요청된 함수 실행 후 사용자가 반환한 결과.


In [27]:
from IPython.display import Markdown, display

def print_history(chat):
  for content in chat.get_history():
      display(Markdown("###" + content.role + ":"))
      for part in content.parts:
          if part.text:
              display(Markdown(part.text))
          if part.function_call:
              print("Function call: {", part.function_call, "}")
          if part.function_response:
              print("Function response: {", part.function_response, "}")
      print("-" * 80)

print_history(chat)

###user:

It's awful dark in here...

--------------------------------------------------------------------------------


###model:

Function call: { id=None args={} name='enable_lights' }
--------------------------------------------------------------------------------


###user:

Function response: { id=None name='enable_lights' response={'result': None} }
--------------------------------------------------------------------------------


###model:

No problem, I can turn the lights on for you.


--------------------------------------------------------------------------------


이 기록은 다음 흐름을 보여줍니다:

1. **사용자**: 메시지 전송.

2. **모델**: 텍스트가 아닌 `enable_lights`를 요청하는 `FunctionCall`로 응답.

3. **사용자(SDK)**: `automatic_function_calling`이 활성화되어 있으므로 `ChatSession`이 자동으로 `enable_lights()`를 실행하고 결과를 `FunctionResponse`로 전송.

4. **모델**: 함수 결과("조명 켜짐")를 사용하여 최종 텍스트 응답 작성.

## 자동 함수 실행 (Python SDK 기능)

위에서 보여준 것처럼, Python SDK의 `ChatSession`에는 자동 함수 실행(Automatic Function Execution)이라는 강력한 기능이 있습니다. 활성화된 경우(기본값), 모델이 FunctionCall로 응답하면 SDK가 다음을 수행합니다:

1. 제공된 `tools`에서 해당 Python 함수를 찾습니다.

2. 모델이 제공한 인수로 함수를 실행합니다.

3. 함수의 반환값을 `FunctionResponse`로 모델에 전송합니다.

4. 모델의 최종 응답(보통 텍스트)만 코드에 반환합니다.

이는 일반적인 사용 사례의 워크플로우를 크게 단순화합니다.

**예제: 수학 연산**

In [28]:
from google.genai import types # Ensure types is imported

def add(a: float, b: float):
    """returns a + b."""
    return a + b

def subtract(a: float, b: float):
    """returns a - b."""
    return a - b

def multiply(a: float, b: float):
    """returns a * b."""
    return a * b

def divide(a: float, b: float):
    """returns a / b."""
    if b == 0:
        return "Cannot divide by zero."
    return a / b

operation_tools = [add, subtract, multiply, divide]

chat = client.chats.create(
    model=MODEL_ID,
    config={
        "tools": operation_tools,
        "automatic_function_calling": {"disable": False} # Enabled by default
    }
)

response = chat.send_message(
    "I have 57 cats, each owns 44 mittens, how many mittens is that in total?"
)

print(response.text)

That would be 2508 mittens in total.


In [29]:
print_history(chat)

###user:

I have 57 cats, each owns 44 mittens, how many mittens is that in total?

--------------------------------------------------------------------------------


###model:

Function call: { id=None args={'a': 57, 'b': 44} name='multiply' }
--------------------------------------------------------------------------------


###user:

Function response: { id=None name='multiply' response={'result': 2508} }
--------------------------------------------------------------------------------


###model:

That would be 2508 mittens in total.

--------------------------------------------------------------------------------


자동 실행이 `multiply` 호출을 원활하게 처리했습니다.

## 자동 함수 스키마 선언

Python SDK의 주요 편의 기능 중 하나는 Python 함수에서 필요한 `FunctionDeclaration` 스키마를 자동으로 생성하는 기능입니다. 다음을 검사합니다:

- **함수 이름**: (`func.__name__`)

- **Docstring**: 함수 설명에 사용됩니다.

- **파라미터**: 이름 및 타입 어노테이션(`int`, `str`, `float`, `bool`, `list`, `dict`). 특정 형식(예: Google 스타일)의 파라미터 docstring도 설명을 향상시킬 수 있습니다.

- **반환 타입 어노테이션**: 모델이 어떤 함수를 호출할지 결정하는 데 엄격하게 사용되지는 않지만, 좋은 관행입니다.

Python 함수를 도구로 직접 사용할 때는 일반적으로 `FunctionDeclaration` 객체를 수동으로 만들 필요가 없습니다.

그러나 스키마를 검사하거나 수정해야 하거나, Python 함수 객체를 쉽게 사용할 수 없는 시나리오에서 사용해야 하는 경우 `genai.types.FunctionDeclaration.from_callable`을 사용하여 명시적으로 스키마를 생성할 수 있습니다.

In [30]:
import json

set_color_declaration = types.FunctionDeclaration.from_callable(
    callable = set_light_color,
    client = client
)

print(json.dumps(set_color_declaration.to_json_dict(), indent=4))

{
    "description": "Set the light color. Lights must be enabled for this to work.",
    "name": "set_light_color",
    "parameters": {
        "properties": {
            "rgb_hex": {
                "type": "STRING"
            }
        },
        "required": [
            "rgb_hex"
        ],
        "type": "OBJECT"
    }
}


## 수동 함수 호출

더 많은 제어가 필요하거나 자동 함수 호출이 불가능한 경우 모델의 [`genai.types.FunctionCall`](https://googleapis.github.io/python-genai/genai.html#genai.types.FunctionCall) 요청을 직접 처리할 수 있습니다. 다음과 같은 경우에 해당합니다:

- 기본값인 `"automatic_function_calling": {"disable": True}`로 `Chat`을 사용하는 경우.
- [`Client.model.generate_content`](https://googleapis.github.io/python-genai/genai.html#genai.types.) 를 사용하는 경우(채팅 기록을 직접 관리).

**예제: 영화**

다음 예제는 Python에서 [함수 호출 단일 턴 curl 샘플](https://ai.google.dev/docs/function_calling#function-calling-single-turn-curl-sample)과 대략적으로 동일합니다. 가상의 API에서 영화 상영 시간 정보를 반환하는(모의) 함수를 사용합니다:

In [31]:
def find_movies(description: str, location: str):
    """find movie titles currently playing in theaters based on any description, genre, title words, etc.

    Args:
        description: Any kind of description including category or genre, title words, attributes, etc.
        location: The city and state, e.g. San Francisco, CA or a zip code e.g. 95616
    """
    return ["Barbie", "Oppenheimer"]


def find_theaters(location: str, movie: str):
    """Find theaters based on location and optionally movie title which are currently playing in theaters.

    Args:
        location: The city and state, e.g. San Francisco, CA or a zip code e.g. 95616
        movie: Any movie title
    """
    return ["Googleplex 16", "Android Theatre"]


def get_showtimes(location: str, movie: str, theater: str, date: str):
    """
    Find the start times for movies playing in a specific theater.

    Args:
      location: The city and state, e.g. San Francisco, CA or a zip code e.g. 95616
      movie: Any movie title
      thearer: Name of the theater
      date: Date for requested showtime
    """
    return ["10:00", "11:00"]

theater_functions = [find_movies, find_theaters, get_showtimes]

`generate_content()`를 사용하여 질문한 후, 모델은 `function_call`을 요청합니다:

In [32]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents="Which theaters in Mountain View, CA show the Barbie movie?",
    config = {
        "tools": theater_functions,
        "automatic_function_calling": {"disable": True}
    }
)

print(json.dumps(response.candidates[0].content.parts[0].to_json_dict(), indent=4))

{
    "function_call": {
        "args": {
            "location": "Mountain View, CA",
            "movie": "Barbie"
        },
        "name": "find_theaters"
    }
}


`ChatSession`과 자동 함수 호출을 사용하지 않으므로 함수를 직접 호출해야 합니다.

간단한 방법은 `if` 문을 사용하는 것입니다:

```python
if function_call.name == 'find_theaters':
  find_theaters(**function_call.args)
elif ...
```

그러나 이미 `theater_functions` 목록을 만들었으므로 다음과 같이 단순화할 수 있습니다:


In [33]:
def call_function(function_call, functions):
    function_name = function_call.name
    function_args = function_call.args
    # Find the function object from the list based on the function name
    for func in functions:
        if func.__name__ == function_name:
            return func(**function_args)

part = response.candidates[0].content.parts[0]

# Check if it's a function call; in real use you'd need to also handle text
# responses as you won't know what the model will respond with.
if part.function_call:
    result = call_function(part.function_call, theater_functions)

print(result)

['Googleplex 16', 'Android Theatre']


마지막으로 다음 `generate_content()` 호출에 응답과 메시지 기록을 전달하여 모델에서 최종 텍스트 응답을 받습니다. 다음 코드 셀은 의도적으로 `Content`를 작성하는 다양한 방법을 보여주므로 원하는 방식을 선택할 수 있습니다.

In [34]:
from google.genai import types
# Build the message history
messages = [
    types.Content(
        role="user",
        parts=[
            types.Part(
                text="Which theaters in Mountain View show the Barbie movie?."
            )
        ]
    ),
    types.Content(
        role="model",
        parts=[part]
    ),
    types.Content(
        role="tool",
        parts=[
            types.Part.from_function_response(
                name=part.function_call.name,
                response={"output":result},
            )
        ]
    )
]

# Generate the next response
response = client.models.generate_content(
    model=MODEL_ID,
    contents=messages,
    config = {
        "tools": theater_functions,
        "automatic_function_calling": {"disable": True}
    }
)
print(response.text)

The Barbie movie is currently playing at the Googleplex 16 and Android Theatre in Mountain View.


이것은 수동 워크플로우를 보여줍니다: 호출, 확인, 실행, 응답, 재호출.

## 병렬 함수 호출

Gemini API는 단일 턴에서 여러 함수를 호출할 수 있습니다. 이는 작업을 완료하기 위해 서로 독립적으로 수행될 수 있는 여러 함수 호출이 있는 시나리오에 적합합니다.

먼저 도구를 설정합니다. 위의 영화 예제와 달리, 이 함수들은 호출되기 위해 서로의 입력이 필요하지 않으므로 병렬 호출의 좋은 후보가 됩니다.

In [35]:
def power_disco_ball(power: bool) -> bool:
    """Powers the spinning disco ball."""
    print(f"Disco ball is {'spinning!' if power else 'stopped.'}")
    return True

def start_music(energetic: bool, loud: bool, bpm: int) -> str:
    """Play some music matching the specified parameters.

    Args:
      energetic: Whether the music is energetic or not.
      loud: Whether the music is loud or not.
      bpm: The beats per minute of the music.

    Returns: The name of the song being played.
    """
    print(f"Starting music! {energetic=} {loud=}, {bpm=}")
    return "Never gonna give you up."


def dim_lights(brightness: float) -> bool:
    """Dim the lights.

    Args:
      brightness: The brightness of the lights, 0.0 is off, 1.0 is full.
    """
    print(f"Lights are now set to {brightness:.0%}")
    return True

house_fns = [power_disco_ball, start_music, dim_lights]

이제 지정된 모든 도구를 사용할 수 있는 지시문으로 모델을 호출합니다.

In [36]:
# You generally set "mode": "any" to make sure Gemini actually *uses* the given tools.
party_chat = client.chats.create(
    model=MODEL_ID,
    config={
        "tools": house_fns,
        "tool_config" : {
            "function_calling_config": {
                "mode": "any"
            }
        }
    }
)

# Call the API
response = party_chat.send_message(
    "Turn this place into a party!"
)


print_history(party_chat)

Disco ball is spinning!
Starting music! energetic=True loud=True, bpm=130
Lights are now set to 30%
Disco ball is spinning!
Starting music! energetic=True loud=True, bpm=130
Lights are now set to 30%
Disco ball is spinning!
Starting music! energetic=True loud=True, bpm=130
Lights are now set to 30%
Disco ball is spinning!
Starting music! energetic=True loud=True, bpm=130
Lights are now set to 30%
Disco ball is spinning!
Starting music! energetic=True loud=True, bpm=130
Lights are now set to 30%
Disco ball is spinning!
Starting music! energetic=True loud=True, bpm=130
Lights are now set to 30%
Disco ball is spinning!
Starting music! energetic=True loud=True, bpm=130
Lights are now set to 30%
Disco ball is spinning!
Starting music! energetic=True loud=True, bpm=130
Lights are now set to 30%
Disco ball is spinning!
Starting music! energetic=True loud=True, bpm=130
Lights are now set to 30%
Disco ball is spinning!
Starting music! energetic=True loud=True, bpm=130
Lights are now set to 30%


###user:

Turn this place into a party!

--------------------------------------------------------------------------------


###model:

...

Function call: { id=None args={'power': True} name='power_disco_ball' }
Function call: { id=None args={'bpm': 130, 'loud': True, 'energetic': True} name='start_music' }
Function call: { id=None args={'brightness': 0.3} name='dim_lights' }
--------------------------------------------------------------------------------


###user:

Function response: { id=None name='power_disco_ball' response={'result': True} }
Function response: { id=None name='start_music' response={'result': 'Never gonna give you up.'} }
Function response: { id=None name='dim_lights' response={'result': True} }
--------------------------------------------------------------------------------


###model:

Function call: { id=None args={'power': True} name='power_disco_ball' }
Function call: { id=None args={'bpm': 130, 'loud': True, 'energetic': True} name='start_music' }
Function call: { id=None args={'brightness': 0.3} name='dim_lights' }
--------------------------------------------------------------------------------


단일 모델 턴에 세 개의 FunctionCall 파트가 포함되어 있으며, SDK가 최종 텍스트 응답을 받기 전에 이를 실행했음을 확인할 수 있습니다.

## 조합형 함수 호출
모델은 여러 턴에 걸쳐 함수 호출을 연결하여, 한 호출의 결과를 다음 호출에 활용할 수 있습니다. 이를 통해 복잡한 다단계 추론 및 작업 완료가 가능합니다.

**예제: 특정 영화 상영 시간 찾기**

`theater_functions`를 재사용하여 영화 검색, 상영관 검색, 상영 시간 검색이 필요한 더 복잡한 쿼리를 요청해 보겠습니다.

In [37]:
chat = client.chats.create(
    model = MODEL_ID,
    config = {
        "tools": theater_functions,
    }
)

response = chat.send_message("""
  Find comedy movies playing in Mountain View, CA on 01/01/2025.
  First, find the movie titles.
  Then, find the theaters showing those movies.
  Finally, find the showtimes for each movie at each theater.
"""
)

print(response.text)
print("\n--- History ---")
print_history(chat)

OK. I have found comedy movies playing in Mountain View, CA on 01/01/2025, the theaters they are playing in, and their showtimes. The comedy movies playing on 01/01/2025 in Mountain View, CA are Barbie and Oppenheimer.

Barbie is playing at Googleplex 16 and Android Theatre at 10:00 and 11:00.
Oppenheimer is playing at Googleplex 16 and Android Theatre at 10:00 and 11:00.

--- History ---


###user:


  Find comedy movies playing in Mountain View, CA on 01/01/2025.
  First, find the movie titles.
  Then, find the theaters showing those movies.
  Finally, find the showtimes for each movie at each theater.


--------------------------------------------------------------------------------


###model:

...

###user:

Function response: { id=None name='get_showtimes' response={'result': ['10:00', '11:00']} }
Function response: { id=None name='get_showtimes' response={'result': ['10:00', '11:00']} }
Function response: { id=None name='get_showtimes' response={'result': ['10:00', '11:00']} }
Function response: { id=None name='get_showtimes' response={'result': ['10:00', '11:00']} }
--------------------------------------------------------------------------------


###model:

OK. I have found comedy movies playing in Mountain View, CA on 01/01/2025, the theaters they are playing in, and their showtimes. The comedy movies playing on 01/01/2025 in Mountain View, CA are Barbie and Oppenheimer.

Barbie is playing at Googleplex 16 and Android Theatre at 10:00 and 11:00.
Oppenheimer is playing at Googleplex 16 and Android Theatre at 10:00 and 11:00.

--------------------------------------------------------------------------------


모델이 질문에 답하기 위해 7번의 호출을 했고, 이후 호출 및 최종 답변에 해당 출력을 활용했음을 확인할 수 있습니다.

## 모드를 사용한 함수 호출 구성

AUTO 모드(또는 SDK의 기본 자동 실행)가 종종 충분하지만, 모델 초기화 또는 `send_message` 중 `tool_config` 파라미터를 사용하여 모델이 함수를 호출할 수 있는 시기와 대상을 정밀하게 제어할 수 있습니다.

`tool_config`는 `FunctionCallingConfig`가 포함된 `ToolConfig` 객체를 허용합니다.

`FunctionCallingConfig`에는 두 가지 주요 필드가 있습니다:

- `mode`: 전체 함수 호출 동작을 제어합니다(AUTO, ANY, NONE).

- `allowed_function_names`: 이 턴에서 모델이 호출할 수 있도록 제한된 함수 이름의 선택적 목록.

### AUTO (기본 모드)

- 동작: 모델이 제공된 `tools`에서 텍스트로 응답할지 또는 하나 이상의 함수를 호출할지 결정합니다. 가장 유연한 모드입니다.

- SDK 기본값: 자동 실행이 활성화된 `ChatSession`을 사용할 때 `tool_config`로 재정의하지 않는 한 기본적으로 `AUTO` 모드를 사용합니다.

In [38]:
chat = client.chats.create(model=MODEL_ID)

response = chat.send_message(
    message="Turn on the lights!",
    config={
        "system_instruction": instruction,
        "tools": light_controls,
        "tool_config" : types.ToolConfig(
            function_calling_config=types.FunctionCallingConfig(
                mode="auto"
            )
        )
    }
)

print_history(chat)

LIGHTBOT: Lights enabled.


###user:

Turn on the lights!

--------------------------------------------------------------------------------


###model:

...

###user:

Function response: { id=None name='enable_lights' response={'result': None} }
--------------------------------------------------------------------------------


###model:

OK. I have turned on the lights.


--------------------------------------------------------------------------------


### NONE 모드
동작: 도구가 제공되더라도 모델이 함수를 호출하는 것을 명시적으로 금지합니다. 텍스트로만 응답합니다. 순수한 대화형 응답을 원하는 턴에 유용합니다.

In [39]:
none_chat = client.chats.create(model=MODEL_ID)

response = none_chat.send_message(
    message="Hello light-bot, what can you do?",
    config={
        "system_instruction": instruction,
        "tools": light_controls, # Tools are provided
        "tool_config" : types.ToolConfig(
            function_calling_config=types.FunctionCallingConfig(
                mode="none"
            )
        ) # but NONE mode prevents their use
    }
)

print_history(none_chat)

###user:

Hello light-bot, what can you do?

--------------------------------------------------------------------------------


###model:

Hello! I can help you with your lights. I can:
*   Turn lights on
*   Turn lights off
*   Set the color of the lights

--------------------------------------------------------------------------------


### ANY 모드
- 동작: 모델이 최소 하나의 함수를 호출하도록 강제합니다.

  - `allowed_function_names`가 설정된 경우, 모델은 해당 목록에서 하나 이상의 함수를 선택해야 합니다.

  - `allowed_function_names`가 설정되지 않은 경우, 모델은 전체 `tools` 목록에서 하나 이상의 함수를 선택해야 합니다.

- 자동 함수 호출이 활성화된 경우 SDK는 [maximum_remote_calls](https://googleapis.github.io/python-genai/genai.html#genai.types.AutomaticFunctionCallingConfig.maximum_remote_calls)에 도달할 때까지 자동으로 함수를 호출합니다(기본값: 10).
- x번의 자동 함수 호출을 허용하려면 `maximum_remote_calls`를 x + 1로 설정합니다. [더 알아보기](https://pypi.org/project/google-genai/#:~:text=Function%20calling%20with%20ANY%20tools%20config%20mode)
- 사용 사례: 애플리케이션 상태가 다음 단계에 특정 작업이나 일련의 작업이 포함되어야 함을 나타낼 때 유용합니다.

In [40]:
chat = client.chats.create(model=MODEL_ID)

response = chat.send_message(
    "Make this place PURPLE!",
    config={
        "system_instruction": instruction,
        "tools": light_controls, # Provide all tools
        "tool_config" : {
            "function_calling_config": {
                "mode": "any"
            }
        },
        "automatic_function_calling": {
            "maximum_remote_calls" : 1
        }
      } # But restrict to available_fns with ANY mode
)

print_history(chat)

LIGHTBOT: Lights enabled.


###user:

Make this place PURPLE!

--------------------------------------------------------------------------------


###model:

Function call: { id=None args={} name='enable_lights' }
--------------------------------------------------------------------------------


###user:

Function response: { id=None name='enable_lights' response={'result': None} }
--------------------------------------------------------------------------------


###model:

Function call: { id=None args={} name='enable_lights' }
--------------------------------------------------------------------------------


## 다음 단계
### 유용한 API 레퍼런스:

- [genai.Client](https://googleapis.github.io/python-genai/genai.html#module-genai.client) 클래스
  - 해당 [Client.models.generate_content](https://googleapis.github.io/python-genai/genai.html#genai.models.Models.generate_content) 메서드에는 특히 도구 및 함수 호출 설정에 사용되는 [genai.types.GenerateContentConfig](https://googleapis.github.io/python-genai/genai.html#genai.types.GenerateContentConfig) 필드가 있습니다.
    - 구성의 `tools` 속성에는 [genai.types.Tools](https://googleapis.github.io/python-genai/genai.html#genai.types.Tool) 객체 목록이 포함됩니다.
    - `function_declarations` 속성에는 [genai.types.FunctionDeclarations](https://googleapis.github.io/python-genai/genai.html#genai.types.FunctionDeclaration) 객체 목록이 포함됩니다.
- [응답](https://googleapis.github.io/python-genai/genai.html#genai.types.GenerateContentResponse)의 [후보](https://googleapis.github.io/python-genai/genai.html#genai.types.Candidate) [콘텐츠](https://googleapis.github.io/python-genai/genai.html#genai.types.Content) [파트](https://googleapis.github.io/python-genai/genai.html#genai.types.Part)에 [genai.types.FunctionCall](https://googleapis.github.io/python-genai/genai.html#genai.types.FunctionCall)이 포함될 수 있으며, `response.candidates[0].contents.parts[0]`에 있습니다.
- `automatic_function_calling`이 비활성화되지 않은 경우, [genai.Chats](https://googleapis.github.io/python-genai/genai.html#module-genai.chats) 세션이 호출을 실행하고 [genai.types.FunctionResponse](https://googleapis.github.io/python-genai/genai.html#genai.types.FunctionResponse)를 전송합니다.
- [FunctionCall](https://googleapis.github.io/python-genai/genai.html#genai.types.FunctionCall)에 대한 응답으로 모델은 항상 [FunctionResponse](https://googleapis.github.io/python-genai/genai.html#genai.types.FunctionResponse)를 기대합니다.
- [Chats.send_message](https://googleapis.github.io/python-genai/genai.html#genai.chats.AsyncChat.send_message) 또는 [models.generate_content](https://googleapis.github.io/python-genai/genai.html#genai.models.Models.generate_content)를 사용하여 수동으로 응답하는 경우, API는 상태가 없으므로 `FunctionResponse`가 포함된 마지막 항목만이 아닌 전체 대화 기록([Content](https://googleapis.github.io/python-genai/genai.html#genai.types.Content) 객체 목록)을 보내야 합니다.

### 관련 예제

함수 호출 사용 아이디어를 더 얻기 위해 이 예제들을 확인하세요:
* [바리스타 봇](../examples/Agents_Function_Calling_Barista_Bot.ipynb): 커피 주문을 처리하는 에이전트
* [브라우저-도구](../examples/Browser_as_a_tool.ipynb): 함수 호출로 웹 브라우저 호출하기.
* 함수 호출로 [검색 결과 재순위](../examples/Search_reranking_using_embeddings.ipynb)하기.
* [Live API에서 도구 사용](../quickstarts/Get_started_LiveAPI_tools.ipynb): Live API에서 함수 호출 및 기타 도구 사용.

### Gemini API 계속 탐색하기

[함수 호출 구성](../quickstarts/Function_calling_config.ipynb) 빠른 시작에서 Gemini API가 함수와 상호작용하는 방식을 제어하는 방법을 배우고, [JSON](../quickstarts/JSON_mode.ipynb) 또는 [Enum](../quickstarts/Enum.ipynb)을 사용하여 모델 출력을 제어하는 방법을 알아보거나, [코드 실행](../quickstarts/Code_Execution.ipynb)을 통해 Gemini API가 직접 코드를 생성하고 실행하는 방법을 배우세요.